# Frozen Perch + BiGRU

One centered focal clip → one feature vector. One soundscape minute → 12 ordered vectors.
LayerNorm → bidirectional GRU (128 units per direction) → dropout → classification head.

Run data preparation first and use the project kernel:
`uv sync --extra notebooks --extra deep-learning --extra audio --extra perch`.

The existing Perch cache is reused. Recording context adds 53 windows for the current
prepared dataset; only missing entries are extracted. Keep the cache writable for that first run.
Missing annotations and focal padding are excluded from loss and AUC.


In [6]:
import json

from birdclef.config import LoaderConfig, RuntimeConfig, TrainConfig, get_config
from birdclef.models import RecurrentExperimentConfig, RecurrentModelConfig
from birdclef.models.encoders.download import download_perch
from birdclef.runtime import apply_environment_defaults

apply_environment_defaults()
defaults = get_config(verbose=False)

RUN_NAME = "perch/bilstm-richer-256"
SMOKE_RUN = False
WANDB_OFFLINE = False  # False enables online W&B with your configured credentials.
CACHE_DIR = defaults.data.cache_dir
ENCODER_PATH = CACHE_DIR / "models/perch_v2_backbone.onnx"
PREPARED_DIR = defaults.data.output_dir / "data_prep"


Detected environment: local


The sampler now draws sequences. A soundscape sequence has several labeled windows,
while a focal sequence has one. Start with soundscape probability 0.01 (approximately
10% labeled windows for this dataset). This is a starting point, not a measured optimum.

Smoke mode preserves whole selected soundscape recordings and validates the full fold.
Use a new run name for changed settings; smoke and full runs have separate folders.


In [7]:
download_perch(ENCODER_PATH)

config = RecurrentExperimentConfig(
    model=RecurrentModelConfig(
        encoder_path=ENCODER_PATH,
        recurrent_dim=256,
        cell="lstm",
        bidirectional=True,
        dropout=0.5,
        head_lr=1e-3,
    ),
    runtime=RuntimeConfig(accelerator="auto", devices=1, precision="32-true"),
    loader=LoaderConfig(
        batch_size=128, val_batch_size=16, predict_batch_size=32,
        num_workers=0, seed=42, soundscape_probability=0.1,
    ),
    train=TrainConfig(
        max_epochs=1 if SMOKE_RUN else 20,
        seed=42, checkpoint_metric="val/auc", checkpoint_mode="max",
        early_stopping_patience=5, gradient_clip_val=1.0,
    ),
    fold=0,
    smoke=SMOKE_RUN,
    data=defaults.data,
    tracking=defaults.tracking.model_copy(update={"offline": WANDB_OFFLINE}),
)
config.data.cache_dir = CACHE_DIR
run_dir = config.data.output_dir / config.run_name(RUN_NAME)
config_path = config.data.output_dir / "configs" / f"{config.run_name(RUN_NAME).replace('/', '-')}.json"
config_path.parent.mkdir(parents=True, exist_ok=True)
config_path.write_text(json.dumps(config.safe_dump(), indent=2), encoding="utf-8")
config.model.show()


                                  RecurrentModelConfig                                  
┏━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Setting          ┃ Value                                                             ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ encoder          │ perch                                                             │
│ encoder_path     │ C:\_GitHub\birdclef-2026\data\cache\models\perch_v2_backbone.onnx │
│ encoder_device   │ cpu                                                               │
│ pretrained       │ True                                                              │
│ hidden_dim       │ None                                                              │
│ dropout          │ 0.5                                                               │
│ encoder_lr       │ 3e-05                                                             │
│ head_lr          │ 0.001                                                             │
│ weight_decay     │ 0.01                                                              │
│ unfreeze_epoch   │ None                                                              │
│ unfreeze_last_n  │ None                                                              │
│ recurrent_dim    │ 256                                                               │
│ recurrent_layers │ 1                                                                 │
│ cell             │ lstm                                                              │
│ bidirectional    │ True                                                              │
└──────────────────┴───────────────────────────────────────────────────────────────────┘

In [8]:
%run -m birdclef.models.fit -- --config "{config_path}" --name "{RUN_NAME}" --prepared-dir "{PREPARED_DIR}"

29-10-2026 18:50:33  INFO      [models/fit:66]  Run: perch/bilstm-richer-256; training examples: 35769; validation windows: 149


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.forge.coreweave.com.
wandb: Appending key for api.forge.coreweave.com to your netrc file: C:\Users\dennm\_netrc


Seed set to 42


Output()

Output()

...

Output()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Restoring states from the checkpoint path at C:\_GitHub\birdclef-2026\outputs\perch\bilstm-richer-256\checkpoints\epoch005.ckpt
c:\_GitHub\birdclef-2026\.venv\Lib\site-packages\lightning\pytorch\trainer\call.py:283: Be aware that when using `ckpt_path`, callbacks used to create the checkpoint need to be provided during `Trainer` instantiation. Please add the following callbacks: ["EarlyStopping{'monitor': 'val/auc', 'mode': 'max'}", "ModelCheckpoint{'monitor': 'val/auc', 'mode': 'max', 'every_n_train_steps': 0, 'every_n_epochs': 1, 'train_time_interval': None}", "ModelCheckpoint{'monitor': None, 'mode': 'min', 'every_n_train_steps': 0, 'every_n_epochs': 1, 'train_time_interval': No

Predicting: |          | 0/? [00:00<?, ?it/s]

best/val_auc,▁
epoch,▁▁▁▁▁▂▂▂▂▂▂▂▂▂▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇██
lr-AdamW/encoder,▁▁▁▁▁▁▁▁▁▁▁
lr-AdamW/head,▁▁▁▁▁▁▁▁▁▁▁
perf/fit_seconds,▁
train/loss_epoch,█▃▂▂▂▁▁▁▁▁▁
train/loss_step,██▅▄▄▂▂▂▃▂▂▂▂▁▁▂▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
trainer/global_step,▁▁▁▁▁▂▂▂▂▂▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇██
val/auc,▁▂▃▆▇█▆▆▅▇▄
val/auc_classes,▁▁▁▁▁▁▁▁▁▁▁
+1,...


Saved training run: C:\_GitHub\birdclef-2026\outputs\perch\bilstm-richer-256


In [9]:
metrics = json.loads((run_dir / "validation-metrics.json").read_text(encoding="utf-8"))
metrics  # Smoke AUC checks the pipeline; it is not an experiment result.

{'auc': 0.9030879894539859,
 'auc_classes': 28,
 'windows': 149,
 'protocol': 'file-gkf5/fold-0'}

## CPU inference without a training cache

Check one real minute using the saved encoder and recurrent checkpoint.
The prediction code fills all 12 context windows, then returns the requested row IDs.


In [10]:
from birdclef.data import load_audio_split
from birdclef.models import RecurrentClassifier

classes = json.loads((PREPARED_DIR / "classes.json").read_text(encoding="utf-8"))
_, valid = load_audio_split(
    PREPARED_DIR, config.data.competition_dir, fold=config.fold, classes=classes
)
minute = valid.loc[valid.filename.eq(valid.filename.iloc[0])]
model = RecurrentClassifier.from_run(run_dir)
model.config.runtime = RuntimeConfig(accelerator="cpu", devices=1, precision="32-true")
predictions = model.predict(minute)
predictions.iloc[:, :8]


Output()

GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
c:\_GitHub\birdclef-2026\.venv\Lib\site-packages\lightning\pytorch\trainer\setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Restoring states from the checkpoint path at C:\_GitHub\birdclef-2026\outputs\perch\bilstm-richer-256\checkpoints\epoch005.ckpt
c:\_GitHub\birdclef-2026\.venv\Lib\site-packages\lightning\pytorch\trainer\call.py:283: Be aware that when using `ckpt_path`, callbacks used to create the checkpoint need to be provided during `Trainer` instantiation. Please add the following callbacks: ["EarlyStopping{'monitor': 'val/auc', 'mode': 'max'}", "ModelCheckpoint{'monitor': 'val/auc', 'mode': 'max', 'every_n_train_steps': 0, '

Predicting: |          | 0/? [00:00<?, ?it/s]

,row_id,1161364,116570,1176823,1491113,1595929,209233,22930
0,BC2026_Train_0004_S08_20250607_070007_5,0.000327,0.029162,0.000123,0.000220,0.000082,0.000033,0.000075
1,BC2026_Train_0004_S08_20250607_070007_10,0.000093,0.038932,0.000042,0.000014,0.000025,0.000009,0.000021
2,BC2026_Train_0004_S08_20250607_070007_15,0.000046,0.007250,0.000021,0.000025,0.000015,0.000006,0.000010
3,BC2026_Train_0004_S08_20250607_070007_20,0.000056,0.105476,0.000023,0.000080,0.000021,0.000008,0.000014
4,BC2026_Train_0004_S08_20250607_070007_25,0.000040,0.013548,0.000021,0.000057,0.000014,0.000006,0.000008
5,BC2026_Train_0004_S08_20250607_070007_30,0.000042,0.012130,0.000018,0.000065,0.000016,0.000006,0.000010
6,BC2026_Train_0004_S08_20250607_070007_35,0.000049,0.014433,0.000017,0.000080,0.000015,0.000006,0.000010
7,BC2026_Train_0004_S08_20250607_070007_40,0.000048,0.013546,0.000014,0.000091,0.000014,0.000006,0.000009
8,BC2026_Train_0004_S08_20250607_070007_45,0.000054,0.011557,0.000015,0.000029,0.000013,0.000005,0.000008
9,BC2026_Train_0004_S08_20250607_070007_50,0.000055,0.009368,0.000012,0.000023,0.000015,0.000006,0.000008


For full training, set `SMOKE_RUN=False` and run the notebook again.
To compare LSTM, change `cell="lstm"` and use a different `RUN_NAME`.

Export a trained run with:
`python -m birdclef.submission.export --run outputs/perch/bigru --all`.

Use the existing Kaggle inference notebook with the exported run and a freshly rebuilt wheelhouse.
No cache Dataset or internet is needed during inference.
